In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix
import plotly.express as px


In [2]:
%pip install matplotlib seaborn plotly scikit-learn pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
df = pd.read_csv("../../week10_students.csv")
print(df.shape)
print(df[["quiz_avg", "assignments_avg", "attendance_pct", "score"]].corr()["score"])

(150, 9)
quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
score              1.000000
Name: score, dtype: float64


In [4]:
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
y = df["score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Train:", len(X_train), "Test:", len(X_test))
# 25% is hidden so we can test the model on students it has never seen.

Train: 112 Test: 38


In [5]:
model = LinearRegression()
model.fit(X_train, y_train)
for name, coef in zip(features, model.coef_):
    print(name, round(coef, 3))
print("Intercept:", round(model.intercept_, 3))
# One extra quiz point = about X more exam points.

quiz_avg 0.579
assignments_avg 0.365
attendance_pct 0.113
Intercept: -4.428


In [6]:
pred = model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R2:", round(r2_score(y_test, pred), 2))
# MAE: on average, the model's prediction is off by about 3 points.
# R2: the model explains about 89% of why students' scores differ.

MAE: 3.33
R2: 0.89


In [7]:
new_student = pd.DataFrame({"quiz_avg": [70], "assignments_avg": [75], "attendance_pct": [90]})
print("New student:", model.predict(new_student)[0])

me = pd.DataFrame({"quiz_avg": [65], "assignments_avg": [70], "attendance_pct": [85]})
print("Me:", model.predict(me)[0])

New student: 73.66410988729878
Me: 68.37835772453471


In [8]:
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())

yc = df["passed"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, yc, test_size=0.25, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(Xc_train, yc_train)

passed
1    131
0     19
Name: count, dtype: int64


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [9]:
yc_pred = clf.predict(Xc_test)
print("Model accuracy:", accuracy_score(yc_test, yc_pred))
print("Lazy baseline:", yc_test.mean())
print(confusion_matrix(yc_test, yc_pred))
# No, the model is not useless. Both accuracies are 0.89, but the lazy baseline
# predicts "pass" for everyone, so it catches 0 of the 4 failing students.
# Our confusion matrix shows the model correctly caught 3 of the 4 students who fail,
# and it made 3 false alarms (said fail, but the student passed).
# Accuracy hides this because most students pass. Catching at-risk students is the real goal.

Model accuracy: 0.8947368421052632
Lazy baseline: 0.8947368421052632
[[ 3  1]
 [ 3 31]]


In [10]:
def test_r2(X_):
    Xtr, Xte, ytr, yte = train_test_split(X_, y, test_size=0.25, random_state=42)
    m = LinearRegression().fit(Xtr, ytr)
    return r2_score(yte, m.predict(Xte))

r2_quiz = test_r2(df[["quiz_avg"]])
r2_all = test_r2(df[features])
r2_prog = test_r2(pd.get_dummies(df[features + ["program"]], columns=["program"], drop_first=True))

print(pd.DataFrame({"Version": ["quiz_avg only", "3 features", "3 features + program"],
                    "R2": [round(r2_quiz, 2), round(r2_all, 2), round(r2_prog, 2)]}))
# What helped: adding assignments and attendance (R2 went from 0.62 to 0.89).
# What didn't: adding program (R2 stayed about the same, 0.88).
# Lesson: more data only helps if it is relevant. Extra features can just add noise.

                Version    R2
0         quiz_avg only  0.62
1            3 features  0.89
2  3 features + program  0.88


In [18]:
import plotly.io as pio
pio.renderers.default = "iframe"

In [19]:
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

In [20]:
fig2 = px.scatter(df, x="quiz_avg", y="score", color=df["passed"].astype(str), hover_name="name", title="2. Who is at risk")
fig2.show()

In [21]:
district_avg = df.groupby("district", as_index=False)["score"].mean()
fig3 = px.bar(district_avg, x="district", y="score", title="3. Where to act")
fig3.show()

In [22]:
fig2.write_html("class_dashboard.html")

In [24]:
fig4 = px.box(df, x="program", y="score", color="program", title="One chart, four stories")
fig4.show()
# Networking is the most consistent program, because it has the shortest box (scores are close together).
# Information Systems has the widest spread, because it has the tallest box and the longest range, from about 29 to 84.

## 6. Model Card

### Purpose
This model predicts a student's score based on quiz average, assignments average, and attendance percentage.

### Features
- quiz_avg
- assignments_avg
- attendance_pct

### Performance
The Linear Regression model achieved an MAE of about 3.3 points and an R² of about 0.89 on the test set.
This means the predictions are usually about 3.3 points away from the actual score, and the model explains about 89% of the variation in scores.

### Risks and Proper Use
The model should be used as a support tool, not as the only basis for decisions about students.
Predictions may be inaccurate for students whose data is different from the training data.